# Absorption and Diffusion

This part is meant to combine all our previous results for absorption, diffusion, and convection shown in the previous three notebooks. The goal is to solve for:

\begin{equation}
\left\{\begin{aligned}
    \partial_t\rho_g&= D(\Delta\rho_g)-\mathbf{v}\cdot(\nabla\rho_g)-\dot m_a(\rho_g, \rho_s) \\
    \partial_t\rho_s&=\dot m_a(\rho_g, \rho_s)
\end{aligned}\right.
\end{equation}
Where $\rho_g$ is the density at that point and $\rho_s$ is the saturation of the metal. $D$ is the diffusivity, and $\mathbf{v}$ the velocity ($\mathbf v=(v_x, v_y)$).

Under initial condition and boundary conditions are (since there is no coupling effects of saturation of the metal I am pretty sure we do not need boundary conditions there, but I'll look into this and phrase it more rigorous):

\begin{equation}
\left\{ \begin{aligned}
    \rho_g(\mathbf{x}, 0)  &= 0, &\forall \mathbf{x}\in \Omega,\\
    \rho_s(\mathbf{x},0) &= 0, &\forall \mathbf{x}\in \Omega,\\
    \rho_g(\mathbf{x}, t)  &= 1, &\forall \mathbf{x}\in\partial\Omega_1,\forall t\geq 0, \\
    \partial_\mathbf{n}\rho_g(\mathbf{x}, t)  &= 0, &\forall \mathbf{x}\in\partial\Omega_2,\forall t\geq 0. \\

\end{aligned}\right.
\end{equation}

where $\mathbf x = (x, y)$. Since the saturation equation contains no spatial derivatives, $\rho_s$ is governed by a local ordinary differential equation and therefore only requires an initial condition, not a boundary condition. The function for calculating adsorption is simplified into:

$$
\dot m(\rho_g, \rho_s) = k\rho_g(\rho_{sat}-\rho_s)
$$

We will be expanding $\hat\rho_g$, $\hat\rho_s$, and $\hat{\dot m}$ (the values at their respective timesteps) into Lagrange polynomials. In our case now we do not yet discretize into time since the solver will do that for us. Previously we would be working with $\rho_g^n$, as indication of time steps, now $\hat{}$ will indicate the last step before time discretization (so the final semi-discrete formulation):

\begin{equation*}
\begin{aligned}
\hat\rho_g(x, t)&=\sum{\hat \rho_g^{i}(t)\psi^i(x)},\\
\hat\rho_s(x, t)&=\sum{\hat \rho_s^{i}(t)\psi^i(x)},\\
\hat{\dot m}(x, t)&=\sum{\hat {\dot m}^{i}(t)\psi^i(x)}.
\end{aligned}
\end{equation*}

we want to solve for $\hat \rho_g$ and $\hat \rho_s$, so our solution vector $\hat {\mathbf{u}}$ becomes:

$$
\hat{\mathbf{u}} = \begin{pmatrix}\hat\rho_g^{1} \\ \vdots \\ \hat\rho_g^{N} \\ \hat\rho_s^{1} \\ \vdots \\ \hat\rho_s^{N}
\end{pmatrix}
$$

We also define the vector $\hat{\mathbf{\dot{m}}}$ by the coefficients of the absorption:

\begin{equation*}
\hat{\mathbf{\dot{m}}}_h =\begin{pmatrix}
\hat m^{1}\\ \vdots \\ \hat m^{N}
\end{pmatrix},\\
\end{equation*}


Our last semi-discrete step becomes now:

$$
M(\partial_t \hat {\mathbf{u}})  = -(K+C)\hat{\mathbf{u}}+
\begin{pmatrix}-M^{gg}\\M^{ss}\end{pmatrix}\hat{\mathbf{\dot{m}}}_h\\
M(\partial_t \hat {\mathbf{u}})=A\hat{\mathbf{u}}+R(\hat{\mathbf{u}})
$$

where (we initially have both $\rho_g$ and $\rho_s$ be the same Lagrange polynomial degree):

\begin{equation*}
\begin{aligned}
M &= 
    \begin{pmatrix}
        M^{gg} & 0 \\ 
        0      & M^{ss}
    \end{pmatrix}\\
M^{gg}_{ij}=M^{ss}_{ij}&=\int \psi^i\psi^j dx\\
K &= \begin{pmatrix}
        K^{gg} & 0 \\ 
        0      & 0
    \end{pmatrix}\\
K^{gg}&=D\int(\nabla \psi^i)(\nabla\psi^j)dx \\
C &= \begin{pmatrix}
        C^{gg} & 0 \\ 
        0      & 0
    \end{pmatrix}\\
C^{gg}_{ij}&=\mathbf{v}\cdot\int(\nabla\psi^i)\psi^jdx\\
\hat{\dot{m}}^i_h&=k\hat\rho_g^i(\rho_{sat}-\hat\rho^i_s)
\end{aligned}
\end{equation*}



and we have the jacobian for the linear part obviously as $A$. Now for the non-linear part we get (by linearity of $M$) $J_{\mathbf{\hat{u}}}(M\hat{\dot{\mathbf{m}}}_h)=M(J_{\mathbf{\hat{u}}}\hat{\dot{\mathbf{m}}}_h)$, which gives us the puzzling:

$$
J_{\mathbf{\hat{u}}}\hat{\dot{\mathbf{m}}}_h=
\begin{pmatrix}
\partial_{\hat{u}^1}\hat{\dot{m}}_h^1 & \partial_{\hat{u}^2}\hat{\dot{m}}_h^1 & \cdots & \partial_{\hat{u}^{N_{bf}}}\hat{\dot{m}}_h^1 \\
\partial_{\hat{u}^1}\hat{\dot{m}}_h^2 & \partial_{\hat{u}^2}\hat{\dot{m}}_h^2 & \cdots & \partial_{\hat{u}^{N_{bf}}}\hat{\dot{m}}_h^2 \\
\vdots & \vdots \ & \ddots & \vdots \\
\partial_{\hat{u}^1}\hat{\dot{m}}_h^{N_{bf}} & \partial_{\hat{u}^2}\hat{\dot{m}}_h^{N_{bf}} & \cdots & \partial_{\hat{u}^{N_{bf}}}\hat{\dot{m}}_h^{N_{bf}} \\
\end{pmatrix}
$$
and because:

$$
\hat{\dot{ m}}^{g,i}_h = -k \hat{u}^{g,i}(\rho_{sat}-\hat{u}^{s,i}) \\
\hat{\dot{ m}}^{s,i}_h = k \hat{u}^{g,i}(\rho_{sat}-\hat{u}^{s,i})
$$

so for each row $i$ corresponding to a $\rho_g$ we get two non zero terms at the columns corresponding to $\hat{u}^{g,i}$, and $\hat{u}^{s,i}$ of values $-k(\rho_{sat}-\hat{u}^{s,i})$, and $k\hat{u}^{g,i}$ respectively. For rows $j$ corresponding to a solid density we get the same values but with flipped signs.


## Import Julia Packages 

In [1]:
using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK, Plots, LinearAlgebra;
using DiffEqBase;
using OrdinaryDiffEqRosenbrock: Rodas5P;

In [ ]:
name = "ab_conv_diff_ODE";

# Stop time, time step
dt = .01;
T = 1.0;

k = 1.0;         # rate of absorption
rho_sat = 1.0;   # saturation density of metal

vx = 1.0;        # x-velocity for convection
vy = 0.0;        # y-velocity for convection
velocity = Vec((vx, vy));
                 # ↑ total velocity 

D = 0.1;         # Diffusivity for diffusion

In [3]:
# generates 20 gridpoints between 
grid = generate_grid(Quadrilateral, (100, 100), Vec((0.0, 0.0)), Vec((1.0, 1.0)));

# generate the same 
qr = QuadratureRule{RefQuadrilateral}(2);

ip_rho = Lagrange{RefQuadrilateral, 1}();
cellvalues_rho = CellValues(qr, ip_rho);

ip_rhos = Lagrange{RefQuadrilateral, 1}()
cellvalues_rhos = CellValues(qr, ip_rhos);

# degree of freedom handler, 
dh = DofHandler(grid)
add!(dh, :rho, ip_rho)
add!(dh, :rho_s, ip_rhos)
close!(dh);

In [4]:
# for now we have no constraints
ch = ConstraintHandler(dh)

#create left_slit
addfacetset!(
    grid,
    "left_slit",
    x -> abs(x[1]) < 1e-8 && 0.25 <= x[2] <= 0.75
)

dbc_left = Dirichlet(
    :rho,
    getfacetset(grid, "left_slit"),
    (x, t) -> 1.0
)

add!(ch, dbc_left)
close!(ch)
update!(ch, 0.0)

In [9]:
function doassemble_M!(M, cellvalues_rho, cellvalues_rhos, dh)
    assembler = start_assemble(M)

    # number of basis functions per field
    n_basefuncs_rho  = getnbasefunctions(cellvalues_rho)
    n_basefuncs_rhos = getnbasefunctions(cellvalues_rhos)

    # loop over all cells
    for cell in CellIterator(dh)
        # get global DOFs for this cell
        cell_dofs = celldofs(cell)
        ndofs_cell = length(cell_dofs)
        Me = zeros(ndofs_cell, ndofs_cell)

        # rho
        Ferrite.reinit!(cellvalues_rho, cell)
        for q in 1:getnquadpoints(cellvalues_rho)
            dx = getdetJdV(cellvalues_rho, q)
            for i in 1:n_basefuncs_rho
                psi_i = shape_value(cellvalues_rho, q, i)
                for j in 1:n_basefuncs_rho
                    psi_j = shape_value(cellvalues_rho, q, j)
                    # :rho DOFs are first in cell_dofs
                    Me[i, j] += psi_i * psi_j * dx
                end
            end
        end

        # rhos
        Ferrite.reinit!(cellvalues_rhos, cell)
        for q in 1:getnquadpoints(cellvalues_rhos)
            dx = getdetJdV(cellvalues_rhos, q)
            for i in 1:n_basefuncs_rhos
                psi_i = shape_value(cellvalues_rhos, q, i)
                for j in 1:n_basefuncs_rhos
                    psi_j = shape_value(cellvalues_rhos, q, j)
                    # :rho_s DOFs are after :rho in cell_dofs
                    offset = n_basefuncs_rho
                    Me[offset + i, offset + j] += psi_i * psi_j * dx
                end
            end
        end

        assemble!(assembler, cell_dofs, Me)
    end

    return M
end

doassemble_M! (generic function with 1 method)

In [10]:
function doassemble_K!(
    K::SparseMatrixCSC, 
    cellvalues_rho::CellValues, 
    cellvalues_rhos::CellValues, 
    dh::DofHandler
    )

    assembler = start_assemble(K)

    # number of basis functions per field
    n_basefuncs_rho  = getnbasefunctions(cellvalues_rho)
    n_basefuncs_rhos = getnbasefunctions(cellvalues_rhos)

    for cell in CellIterator(dh)

        # get global DOFs for this cell
        cell_dofs = celldofs(cell)
        ndofs_cell = length(cell_dofs)
        Ke = zeros(ndofs_cell, ndofs_cell)

        # rho is the only important bit since rho_s does not diffuse
        Ferrite.reinit!(cellvalues_rho, cell)

        for q_point in 1:getnquadpoints(cellvalues_rho)
            dx = getdetJdV(cellvalues_rho, q_point)

            for i in 1:n_basefuncs_rho
                dpsi_i = shape_gradient(cellvalues_rho, q_point, i)

                for j in 1:n_basefuncs_rho
                    dpsi_j = shape_gradient(cellvalues_rho, q_point, j)
                    # D∫ (∇ψ^i) ⋅ (∇ψ^j)dx
                    Ke[i, j] += D * (dpsi_j ⋅ dpsi_i) * dx
                end
            end
        end

        assemble!(assembler, cell_dofs, Ke)
    end

    return K
end


doassemble_K! (generic function with 1 method)

In [11]:
function doassemble_C!(
    K::SparseMatrixCSC, 
    cellvalues_rho::CellValues, 
    cellvalues_rhos::CellValues, 
    dh::DofHandler
    )

    n_basefuncs_rho = getnbasefunctions(cellvalues_rho)
    Ce = zeros(n_basefuncs_rho, n_basefuncs_rho)

    assembler = start_assemble(C)

    for cell in CellIterator(dh)
        
        # get global DOFs for this cell
        cell_dofs = celldofs(cell)
        ndofs_cell = length(cell_dofs)
        Ce = zeros(ndofs_cell, ndofs_cell)

        # rho is the only important bit since rho_s does not diffuse
        Ferrite.reinit!(cellvalues_rho, cell)
        fill!(Ce, 0)

        for q_point in 1:getnquadpoints(cellvalues_rho)
            dx = getdetJdV(cellvalues_rho, q_point)

            for i in 1:n_basefuncs_rho
                psi_i = shape_value(cellvalues_rho, q_point, i)

                for j in 1:n_basefuncs_rho
                    dpsi_j = shape_gradient(cellvalues_rho, q_point, j)
                    # -v \psi^i(\partial_x\psi^j)dx
                    Ce[i, j] += (velocity ⋅ dpsi_j) *  psi_i * dx
                end
            end
        end

        assemble!(assembler, cell_dofs, Ce)
    end

    return C
end


doassemble_C! (generic function with 1 method)

In [12]:
M = allocate_matrix(dh);
K = allocate_matrix(dh);
C = allocate_matrix(dh);

M = doassemble_M!(M, cellvalues_rho, cellvalues_rhos, dh);
K = doassemble_K!(K, cellvalues_rho, cellvalues_rhos, dh);
C = doassemble_C!(C, cellvalues_rho, cellvalues_rhos, dh);

A = -(K+C)

20402×20402 SparseMatrixCSC{Float64, Int64} with 90601 stored entries:
⎡⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [13]:
ndofs_total = ndofs(dh)
u_n = zeros(ndofs_total)

apply!(u_n, ch)
apply!(M, ch);

jac_sparsity = sparse(A)

# rhsdata = get_rhs_data(ch, A);

20402×20402 SparseMatrixCSC{Float64, Int64} with 90601 stored entries:
⎡⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [18]:
jac_sparsity == A

true

In [14]:
rho_dofs  = Int[]
rhos_dofs = Int[]

for cell in CellIterator(dh)
    cdofs = celldofs(cell)

    append!(rho_dofs,  cdofs[dof_range(dh, :rho)])
    append!(rhos_dofs, cdofs[dof_range(dh, :rho_s)])
end

rho_dofs  = unique(rho_dofs);
rhos_dofs = unique(rhos_dofs);

In [17]:
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_rho::CellValues
    cellvalues_rhos::CellValues
    u::Vector
end

In [ ]:
p = RHSparams(K, ch, dh, cellvalues_rho, cellvalues_rhos, copy(u_n))

In [ ]:
pvd = paraview_collection(joinpath(name, name))

VTKGridFile(joinpath(name, name), dh) do vtk
    write_solution(vtk, dh, u_n)
    pvd[0.0] = vtk
end